In [ ]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_curve, auc
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from tqdm import tqdm

In [4]:
# Test kernel
print("Kernel is working!")
import psutil
print(f"Memory usage: {psutil.virtual_memory().percent}%")

Kernel is working!
Memory usage: 41.6%


In [2]:
# --- 1. Dataset Version Used for Training ---
# Confirming dataset source and initial load
print("Loading dataset...")
df = pd.read_parquet("../data/processed/dataset_192H_60_forecast.parquet")
print(f"Dataset loaded with {len(df)} rows and {len(df.columns)} columns.")

Loading dataset...
Dataset loaded with 24571322 rows and 19 columns.


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24571322 entries, 0 to 24571321
Data columns (total 19 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   observed_at              datetime64[ns, UTC]
 1   stop_id                  object             
 2   stop_name                object             
 3   line_id                  object             
 4   vehicle_id               object             
 5   direction                object             
 6   platform_name            object             
 7   destination_name         object             
 8   hour                     int32              
 9   weekday                  int32              
 10  is_weekend               int64              
 11  time_to_station          int64              
 12  roll_mean_tts_10m        float64            
 13  roll_max_tts_10m         float64            
 14  roll_count_10m           float64            
 15  baseline_median_tts      float

In [6]:
df.sample(5)

,observed_at,stop_id,stop_name,line_id,vehicle_id,direction,platform_name,destination_name,hour,weekday,is_weekend,time_to_station,roll_mean_tts_10m,roll_max_tts_10m,roll_count_10m,baseline_median_tts,deviation_from_baseline,late,late_3min
13587070,2026-03-01 21:29:19.777743+00:00,940GZZLUSJW,St. John's Wood Underground Station,jubilee,340,inbound,Northbound - Platform 1,Stanmore Underground Station,21,6,1,1059,997.941032,1788.0,407.0,962.0,97.0,1,0
23985790,2026-03-03 19:55:31.991319+00:00,940GZZLUWWL,Walthamstow Central Underground Station,victoria,232,unknown,Southbound - Platform 2,Walthamstow Central Underground Station,19,1,0,1781,883.726277,1786.0,1096.0,870.0,911.0,1,1
501474,2026-02-26 08:54:43.020466+00:00,940GZZLUBMY,Bermondsey Underground Station,jubilee,133,inbound,Westbound - Platform 1,Willesden Green Underground Station,8,3,0,578,394.650000,850.0,140.0,394.0,184.0,1,1
22315571,2026-03-04 18:20:28.842418+00:00,940GZZLUWRR,Warren Street Underground Station,victoria,211,inbound,Southbound - Platform 4,Brixton Underground Station,18,2,0,606,442.997340,946.0,376.0,426.0,180.0,1,0
16800472,2026-02-26 13:30:46.020979+00:00,940GZZLUSVS,Seven Sisters Underground Station,victoria,242,outbound,Northbound - Platform 3,Walthamstow Central Underground Station,13,3,0,1371,721.201195,1453.0,502.0,710.0,661.0,1,1


In [3]:
counts = df.groupby("vehicle_id").size()
print(counts.describe())

count       160.000000
mean     153570.762500
std      113970.633684
min           5.000000
25%       46786.500000
50%      135597.000000
75%      245546.000000
max      385881.000000
dtype: float64


In [4]:
v = counts.index[0]

tmp = df[df["vehicle_id"] == v].sort_values("observed_at")

tmp[["observed_at", "time_to_station"]].head(20)

,observed_at,time_to_station
1706625,2026-02-24 23:43:28.408498+00:00,1577
11520069,2026-02-24 23:43:28.408498+00:00,286
5759556,2026-02-24 23:43:28.408498+00:00,436
14036538,2026-02-24 23:43:28.408498+00:00,1214
8020169,2026-02-24 23:43:28.408498+00:00,1701
1706626,2026-02-24 23:43:40.931174+00:00,1577
24499558,2026-02-24 23:43:40.931174+00:00,32
14036539,2026-02-24 23:43:40.931174+00:00,1214
21391795,2026-02-24 23:43:40.931174+00:00,601
8020170,2026-02-24 23:43:40.931174+00:00,1701


In [5]:
tmp["dt"] = tmp["observed_at"].diff()

tmp["dt"].describe()

count                        20863
mean     0 days 00:00:28.014896359
std      0 days 00:11:20.279962942
min                0 days 00:00:00
25%                0 days 00:00:00
50%                0 days 00:00:00
75%                0 days 00:00:00
max         0 days 14:55:46.776200
Name: dt, dtype: object

In [6]:
tmp[["observed_at", "time_to_station"]].head(50)

,observed_at,time_to_station
1706625,2026-02-24 23:43:28.408498+00:00,1577
11520069,2026-02-24 23:43:28.408498+00:00,286
5759556,2026-02-24 23:43:28.408498+00:00,436
14036538,2026-02-24 23:43:28.408498+00:00,1214
8020169,2026-02-24 23:43:28.408498+00:00,1701
1706626,2026-02-24 23:43:40.931174+00:00,1577
24499558,2026-02-24 23:43:40.931174+00:00,32
14036539,2026-02-24 23:43:40.931174+00:00,1214
21391795,2026-02-24 23:43:40.931174+00:00,601
8020170,2026-02-24 23:43:40.931174+00:00,1701


In [7]:
keys = ["vehicle_id", "destination_name", "direction"]

counts = df.groupby(keys).size()

print(counts.describe())

count      1136.000000
mean      21618.705106
std       34530.669953
min           1.000000
25%        1059.500000
50%        6837.500000
75%       23248.000000
max      207496.000000
dtype: float64


In [8]:
k = counts.index[0]

tmp = df[
    (df["vehicle_id"] == k[0]) &
    (df["destination_name"] == k[1]) &
    (df["direction"] == k[2])
].sort_values("observed_at")

tmp[["observed_at", "time_to_station"]].head(20)

,observed_at,time_to_station
10703958,2026-02-26 07:56:30.388150+00:00,1741
1426924,2026-02-26 07:56:30.388150+00:00,1159
18065219,2026-02-26 07:56:30.388150+00:00,1694
21224010,2026-02-26 07:56:47.946096+00:00,195
20644367,2026-02-26 07:56:47.946096+00:00,465
18065229,2026-02-26 07:56:47.946096+00:00,1694
9387645,2026-02-26 07:56:47.946096+00:00,362
7147579,2026-02-26 07:56:47.946096+00:00,603
1426934,2026-02-26 07:56:47.946096+00:00,1159
21739805,2026-02-26 07:56:47.946096+00:00,1618


In [9]:
keys = ["vehicle_id", "stop_id", "direction"]

counts = df.groupby(keys).size()

print(counts.describe())

count     7573.000000
mean      3244.595537
std       4072.155007
min          1.000000
25%        508.000000
50%       1595.000000
75%       4890.000000
max      36919.000000
dtype: float64


In [10]:
k = counts.index[0]

tmp = df[
    (df["vehicle_id"] == k[0]) &
    (df["stop_id"] == k[1]) &
    (df["direction"] == k[2])
].sort_values("observed_at")

tmp[["observed_at", "time_to_station"]].head(20)

,observed_at,time_to_station
483917,2026-02-25 10:52:38.850400+00:00,372
483926,2026-02-25 10:53:05.034061+00:00,347
483931,2026-02-25 10:53:18.717095+00:00,365
483936,2026-02-25 10:53:32.452926+00:00,365
483941,2026-02-25 10:53:47.825747+00:00,305
483946,2026-02-25 10:53:59.176819+00:00,305
483951,2026-02-25 10:54:20.520571+00:00,306
496796,2026-02-25 23:07:56.207308+00:00,436
496795,2026-02-25 23:07:56.207308+00:00,256
496797,2026-02-25 23:07:56.207308+00:00,616


In [11]:
keys = ["vehicle_id", "stop_id", "direction", "destination_name"]

df = df.sort_values(keys + ["observed_at"])

In [12]:
df["dt"] = df.groupby(keys)["observed_at"].diff().dt.total_seconds()

In [13]:
GAP = 300

df["new_run"] = (df["dt"].isna()) | (df["dt"] > GAP)

In [14]:
df["run_id"] = df.groupby(keys)["new_run"].cumsum()

In [15]:
k = df[keys].iloc[0].tolist()

tmp = df[
    (df["vehicle_id"] == k[0]) &
    (df["stop_id"] == k[1]) &
    (df["direction"] == k[2]) &
    (df["destination_name"] == k[3])
].sort_values("observed_at")

tmp[["observed_at", "time_to_station", "run_id"]].head(50)

,observed_at,time_to_station,run_id
496793,2026-02-25 23:07:56.207308+00:00,46,1.0
496796,2026-02-25 23:07:56.207308+00:00,436,1.0
496797,2026-02-25 23:07:56.207308+00:00,616,1.0
496806,2026-02-25 23:08:37.968459+00:00,46,1.0
496809,2026-02-25 23:08:37.968459+00:00,436,1.0
496810,2026-02-25 23:08:37.968459+00:00,616,1.0
496812,2026-02-25 23:08:49.965967+00:00,415,1.0
496813,2026-02-25 23:08:49.965967+00:00,595,1.0
496816,2026-02-25 23:09:03.136816+00:00,373,1.0
496817,2026-02-25 23:09:03.136816+00:00,613,1.0


In [16]:
run_counts = df.groupby(
    keys + ["run_id"]
).size()

print(run_counts.describe())

count    265350.000000
mean         92.552663
std         228.116441
min           1.000000
25%          42.000000
50%          82.000000
75%         113.000000
max       15581.000000
dtype: float64


In [22]:
# ----------------------------
# SETTINGS
# ----------------------------
HORIZON_SEC = 120
GROUP_KEYS = ["vehicle_id", "stop_id", "direction", "destination_name", "run_id"]

# make sure sorted correctly
df = df.sort_values(GROUP_KEYS + ["observed_at"]).reset_index(drop=True)

# store original row id so we can merge back later
df["row_id"] = np.arange(len(df))

future_parts = []

for key, g in df.groupby(GROUP_KEYS, sort=False):
    g = g.sort_values("observed_at").copy()

    # target future time for each row
    g["target_time"] = g["observed_at"] + pd.Timedelta(seconds=HORIZON_SEC)

    # prepare right-side table: future candidates within same run
    right = g[["observed_at", "late_3min"]].copy()
    right = right.rename(columns={
        "observed_at": "future_observed_at",
        "late_3min": "future_late_3min_120s"
    })

    # merge_asof: for each target_time, find first future row with time >= target_time
    matched = pd.merge_asof(
        g.sort_values("target_time"),
        right.sort_values("future_observed_at"),
        left_on="target_time",
        right_on="future_observed_at",
        direction="forward"
    )

    future_parts.append(
        matched[["row_id", "target_time", "future_observed_at", "future_late_3min_120s"]]
    )

future_df = pd.concat(future_parts, ignore_index=True)

# merge back to main dataframe
df = df.merge(future_df, on="row_id", how="left")

# optional: how many rows got a future label?
print(df["future_late_3min_120s"].isna().mean(), "fraction missing future label")

# keep only rows where future label exists
df_path_b = df.dropna(subset=["future_late_3min_120s"]).copy()

# make target integer
df_path_b["future_late_3min_120s"] = df_path_b["future_late_3min_120s"].astype(int)

print("Original rows:", len(df))
print("Path B rows with valid future target:", len(df_path_b))
print(df_path_b["future_late_3min_120s"].value_counts(dropna=False))

0.09701118238570965 fraction missing future label
Original rows: 24571322
Path B rows with valid future target: 22187629
future_late_3min_120s
0    15396777
1     6790852
Name: count, dtype: int64


In [17]:
print("Original rows:", len(df))
print("Path B rows with valid future target:", len(df_path_b))
print(df_path_b["future_late_3min_120s"].value_counts(dropna=False))

sample_cols = [
    "observed_at",
    "target_time",
    "future_observed_at",
    "late_3min",
    "future_late_3min_120s",
    "vehicle_id",
    "stop_id",
    "direction",
    "destination_name",
    "run_id"
]

print(df_path_b[sample_cols].head(20))

Original rows: 24571322


NameError: name 'df_path_b' is not defined

In [27]:
df.sample(5)

,observed_at,stop_id,stop_name,line_id,vehicle_id,direction,platform_name,destination_name,hour,weekday,...,deviation_from_baseline,late,late_3min,dt,new_run,run_id,row_id,target_time,future_observed_at,future_late_3min_120s
20773731,2026-03-03 22:13:17.904407+00:00,940GZZLUWIG,Willesden Green Underground Station,jubilee,346,inbound,Northbound - Platform 2,Stanmore Underground Station,22,1,...,-889.0,0,0,12.136651,False,44.0,20773731,2026-03-03 22:15:17.904407+00:00,2026-03-03 22:15:25.424582+00:00,0.0
14916517,2026-03-03 08:42:52.608728+00:00,940GZZLUKBN,Kilburn Underground Station,jubilee,320,inbound,Northbound - Platform 1,Stanmore Underground Station,8,1,...,577.0,1,1,15.491893,False,38.0,14916517,2026-03-03 08:44:52.608728+00:00,2026-03-03 08:44:58.378512+00:00,1.0
6236071,2026-02-26 12:39:00.780455+00:00,940GZZLUSKW,Stockwell Underground Station,victoria,225,outbound,Northbound - Platform 1,Walthamstow Central Underground Station,12,3,...,-31.0,0,0,12.921894,False,21.0,6236071,2026-02-26 12:41:00.780455+00:00,NaT,NaN
17282429,2026-02-25 21:23:00.016210+00:00,940GZZLUGPK,Green Park Underground Station,jubilee,331,inbound,Northbound - Platform 5,Stanmore Underground Station,21,2,...,482.0,1,1,13.843254,False,8.0,17282429,2026-02-25 21:25:00.016210+00:00,2026-02-25 21:25:03.197138+00:00,1.0
20798524,2026-03-03 15:27:09.507192+00:00,940GZZLUWSM,Westminster Underground Station,jubilee,346,inbound,Westbound - Platform 4,Stanmore Underground Station,15,1,...,-37.0,0,0,14.466196,False,41.0,20798524,2026-03-03 15:29:09.507192+00:00,2026-03-03 15:29:21.043843+00:00,0.0


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24571322 entries, 0 to 24571321
Data columns (total 26 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   observed_at              datetime64[ns, UTC]
 1   stop_id                  object             
 2   stop_name                object             
 3   line_id                  object             
 4   vehicle_id               object             
 5   direction                object             
 6   platform_name            object             
 7   destination_name         object             
 8   hour                     int32              
 9   weekday                  int32              
 10  is_weekend               int64              
 11  time_to_station          int64              
 12  roll_mean_tts_10m        float64            
 13  roll_max_tts_10m         float64            
 14  roll_count_10m           float64            
 15  baseline_median_tts      float

In [33]:
df_path_b.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22187629 entries, 17862147 to 3967249
Data columns (total 26 columns):
 #   Column                   Dtype              
---  ------                   -----              
 0   observed_at              datetime64[ns, UTC]
 1   stop_id                  object             
 2   stop_name                object             
 3   line_id                  object             
 4   vehicle_id               object             
 5   direction                object             
 6   platform_name            object             
 7   destination_name         object             
 8   hour                     int32              
 9   weekday                  int32              
 10  is_weekend               int64              
 11  time_to_station          int64              
 12  roll_mean_tts_10m        float64            
 13  roll_max_tts_10m         float64            
 14  roll_count_10m           float64            
 15  baseline_median_tts      floa

In [29]:
df_path_b.sample(5)

,observed_at,stop_id,stop_name,line_id,vehicle_id,direction,platform_name,destination_name,hour,weekday,...,deviation_from_baseline,late,late_3min,dt,new_run,run_id,row_id,target_time,future_observed_at,future_late_3min_120s
5881744,2026-02-28 22:39:49.022698+00:00,940GZZLUBLR,Blackhorse Road Underground Station,victoria,224,outbound,Northbound - Platform 1,Walthamstow Central Underground Station,22,5,...,682.0,1,1,11.657318,False,43.0,5881744,2026-02-28 22:41:49.022698+00:00,2026-02-28 22:41:50.282752+00:00,1
9108627,2026-02-28 14:31:30.464127+00:00,940GZZLUSVS,Seven Sisters Underground Station,victoria,251,outbound,Northbound - Platform 3,Walthamstow Central Underground Station,14,5,...,-83.0,0,0,14.694336,False,1.0,9108627,2026-02-28 14:33:30.464127+00:00,2026-02-28 14:33:34.278778+00:00,0
7378299,2026-02-24 21:35:05.068214+00:00,940GZZLUVIC,Victoria Underground Station,victoria,232,inbound,Southbound - Platform 4,Brixton Underground Station,21,1,...,327.0,1,1,12.830896,False,5.0,7378299,2026-02-24 21:37:05.068214+00:00,2026-02-24 21:37:11.799637+00:00,1
12739203,2026-03-03 23:06:17.769859+00:00,940GZZLUWHM,West Ham Underground Station,jubilee,307,outbound,Eastbound - Platform 6,Stratford Underground Station,23,1,...,-180.5,0,0,12.274045,False,56.0,12739203,2026-03-03 23:08:17.769859+00:00,2026-03-03 23:08:27.455777+00:00,0
15209935,2026-02-26 08:40:35.719191+00:00,940GZZLUFYR,Finchley Road Underground Station,jubilee,321,inbound,Northbound - Platform 2,Wembley Park Underground Station,8,3,...,90.0,1,0,16.811427,False,6.0,15209935,2026-02-26 08:42:35.719191+00:00,2026-02-26 08:42:52.610099+00:00,0


In [31]:
print(df_path_b["observed_at"].min())
print(df_path_b["observed_at"].max())
print(df_path_b["observed_at"].describe())

2026-02-24 16:56:02.356540+00:00
2026-03-04 19:29:36.710039+00:00
count                               22187629
mean     2026-02-28 23:58:59.794386176+00:00
min         2026-02-24 16:56:02.356540+00:00
25%      2026-02-26 23:27:53.850168064+00:00
50%         2026-02-28 21:23:13.293920+00:00
75%      2026-03-02 23:20:54.737896960+00:00
max         2026-03-04 19:29:36.710039+00:00
Name: observed_at, dtype: object


In [32]:
df_path_b = df_path_b.sort_values("observed_at")

t1 = df_path_b["observed_at"].quantile(0.7)
t2 = df_path_b["observed_at"].quantile(0.85)

print("t1 =", t1)
print("t2 =", t2)

train_df = df_path_b[df_path_b["observed_at"] < t1]
val_df   = df_path_b[(df_path_b["observed_at"] >= t1) & (df_path_b["observed_at"] < t2)]
test_df  = df_path_b[df_path_b["observed_at"] >= t2]

print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

t1 = 2026-03-02 15:31:05.135026944+00:00
t2 = 2026-03-03 21:14:56.594266112+00:00
train: 15531335
val: 3328521
test: 3327773


In [43]:
print(df_path_b.columns.tolist())

['observed_at', 'stop_id', 'stop_name', 'line_id', 'vehicle_id', 'direction', 'platform_name', 'destination_name', 'hour', 'weekday', 'is_weekend', 'time_to_station', 'roll_mean_tts_10m', 'roll_max_tts_10m', 'roll_count_10m', 'baseline_median_tts', 'deviation_from_baseline', 'late', 'late_3min', 'dt', 'new_run', 'run_id', 'row_id', 'target_time', 'future_observed_at', 'future_late_3min_120s']


In [44]:
df_path_b.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 22187629 entries, 17862147 to 3967249
Data columns (total 26 columns):
 #   Column                   Non-Null Count     Dtype              
---  ------                   --------------     -----              
 0   observed_at              22187629 non-null  datetime64[ns, UTC]
 1   stop_id                  22187629 non-null  object             
 2   stop_name                22187629 non-null  object             
 3   line_id                  22187629 non-null  object             
 4   vehicle_id               22187629 non-null  object             
 5   direction                22187629 non-null  object             
 6   platform_name            22187629 non-null  object             
 7   destination_name         22187629 non-null  object             
 8   hour                     22187629 non-null  int32              
 9   weekday                  22187629 non-null  int32              
 10  is_weekend               22187629 non-null  int64  

In [45]:
drop_cols = [
    "future_late_3min_120s",
    "future_observed_at",
    "target_time",
    "row_id",
    "run_id",
    "new_run",
    "dt",
    "late",
    "late_3min",
    "observed_at"
]

feature_cols = [c for c in train_df.columns if c not in drop_cols]

print("Features:", feature_cols)
print("N features:", len(feature_cols))

X_train = train_df[feature_cols]
y_train = train_df["future_late_3min_120s"]

X_val = val_df[feature_cols]
y_val = val_df["future_late_3min_120s"]

X_test = test_df[feature_cols]
y_test = test_df["future_late_3min_120s"]

print(X_train.shape, X_val.shape, X_test.shape)

Features: ['stop_id', 'stop_name', 'line_id', 'vehicle_id', 'direction', 'platform_name', 'destination_name', 'hour', 'weekday', 'is_weekend', 'time_to_station', 'roll_mean_tts_10m', 'roll_max_tts_10m', 'roll_count_10m', 'baseline_median_tts', 'deviation_from_baseline']
N features: 16
(15531335, 16) (3328521, 16) (3327773, 16)


In [46]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

# ----------------------------
# TARGET
# ----------------------------
target_col = "future_late_3min_120s"

# ----------------------------
# FEATURE GROUPS
# ----------------------------
categorical_features = [
    "stop_id",
    "stop_name",
    "line_id",
    "vehicle_id",
    "direction",
    "platform_name",
    "destination_name",
]

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

feature_cols = categorical_features + numeric_features

# ----------------------------
# X / y
# ----------------------------
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

Train: (15531335, 16) (15531335,)
Val:   (3328521, 16) (3328521,)
Test:  (3327773, 16) (3327773,)


In [47]:
# ----------------------------
# PREPROCESSORS
# ----------------------------

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

numeric_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer_rf, numeric_features),
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer_lr, numeric_features),
    ]
)

In [49]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_rf),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest...")
rf_pipeline.fit(X_train, y_train)

val_proba_rf = rf_pipeline.predict_proba(X_val)[:, 1]
val_pred_rf = (val_proba_rf >= 0.5).astype(int)

print("\n=== RANDOM FOREST: VALIDATION REPORT ===")
print(classification_report(y_val, val_pred_rf, digits=4))
print("ROC-AUC:", roc_auc_score(y_val, val_proba_rf))
print("PR-AUC :", average_precision_score(y_val, val_proba_rf))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred_rf))

Fitting Random Forest...

=== RANDOM FOREST: VALIDATION REPORT ===
              precision    recall  f1-score   support

           0     0.9889    0.9605    0.9745   2336002
           1     0.9129    0.9746    0.9427    992519

    accuracy                         0.9647   3328521
   macro avg     0.9509    0.9676    0.9586   3328521
weighted avg     0.9662    0.9647    0.9650   3328521

ROC-AUC: 0.9939707244167337
PR-AUC : 0.9860831255255261
Confusion Matrix:
 [[2243661   92341]
 [  25166  967353]]


In [50]:
# =========================
# MODEL V1 (already trained)
# keep these as reference
# =========================
rf_pipeline_v1 = rf_pipeline
val_proba_rf_v1 = val_proba_rf
val_pred_rf_v1 = val_pred_rf

# optional: save metrics in a dict
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

results = {}

results["rf_v1_with_vehicle_id"] = {
    "roc_auc": roc_auc_score(y_val, val_proba_rf_v1),
    "pr_auc": average_precision_score(y_val, val_proba_rf_v1),
    "report": classification_report(y_val, val_pred_rf_v1, digits=4),
    "confusion_matrix": confusion_matrix(y_val, val_pred_rf_v1),
}

print(results["rf_v1_with_vehicle_id"]["report"])
print(results["rf_v1_with_vehicle_id"]["confusion_matrix"])

              precision    recall  f1-score   support

           0     0.9889    0.9605    0.9745   2336002
           1     0.9129    0.9746    0.9427    992519

    accuracy                         0.9647   3328521
   macro avg     0.9509    0.9676    0.9586   3328521
weighted avg     0.9662    0.9647    0.9650   3328521

[[2243661   92341]
 [  25166  967353]]


In [51]:
# =========================
# MODEL V2 (without vehicle_id)
# =========================
categorical_features_v2 = [
    "stop_id",
    "stop_name",
    "line_id",
    "direction",
    "platform_name",
    "destination_name",
]

numeric_features_v2 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

feature_cols_v2 = categorical_features_v2 + numeric_features_v2

X_train_v2 = train_df[feature_cols_v2].copy()
X_val_v2   = val_df[feature_cols_v2].copy()
X_test_v2  = test_df[feature_cols_v2].copy()

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

categorical_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer_v2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor_rf_v2 = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer_v2, categorical_features_v2),
        ("num", numeric_transformer_v2, numeric_features_v2),
    ]
)

rf_pipeline_v2 = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_v2),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest v2 (without vehicle_id)...")
rf_pipeline_v2.fit(X_train_v2, y_train)

val_proba_rf_v2 = rf_pipeline_v2.predict_proba(X_val_v2)[:, 1]
val_pred_rf_v2 = (val_proba_rf_v2 >= 0.5).astype(int)

Fitting Random Forest v2 (without vehicle_id)...


In [52]:
results["rf_v2_without_vehicle_id"] = {
    "roc_auc": roc_auc_score(y_val, val_proba_rf_v2),
    "pr_auc": average_precision_score(y_val, val_proba_rf_v2),
    "report": classification_report(y_val, val_pred_rf_v2, digits=4),
    "confusion_matrix": confusion_matrix(y_val, val_pred_rf_v2),
}

print(results["rf_v2_without_vehicle_id"]["report"])
print(results["rf_v2_without_vehicle_id"]["confusion_matrix"])

              precision    recall  f1-score   support

           0     0.9897    0.9609    0.9751   2336002
           1     0.9140    0.9764    0.9442    992519

    accuracy                         0.9656   3328521
   macro avg     0.9518    0.9687    0.9596   3328521
weighted avg     0.9671    0.9656    0.9659   3328521

[[2244780   91222]
 [  23416  969103]]


In [53]:
print("=== COMPARISON ===")
print("V1 ROC-AUC:", results["rf_v1_with_vehicle_id"]["roc_auc"])
print("V2 ROC-AUC:", results["rf_v2_without_vehicle_id"]["roc_auc"])
print("V1 PR-AUC :", results["rf_v1_with_vehicle_id"]["pr_auc"])
print("V2 PR-AUC :", results["rf_v2_without_vehicle_id"]["pr_auc"])

=== COMPARISON ===
V1 ROC-AUC: 0.9939707244167337
V2 ROC-AUC: 0.9956153748828321
V1 PR-AUC : 0.9860831255255261
V2 PR-AUC : 0.989837546818113


## Checking the same model with numerical features only

In [18]:
numeric_features_v3 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

# ----------------------------
# V3: numeric only
# ----------------------------
numeric_features_v3 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_v3 = train_df[numeric_features_v3].copy()
X_val_v3   = val_df[numeric_features_v3].copy()
X_test_v3  = test_df[numeric_features_v3].copy()

numeric_pipeline_v3 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest v3 (numeric only)...")
numeric_pipeline_v3.fit(X_train_v3, y_train)

val_proba_rf_v3 = numeric_pipeline_v3.predict_proba(X_val_v3)[:, 1]
val_pred_rf_v3 = (val_proba_rf_v3 >= 0.5).astype(int)

print("\n=== RANDOM FOREST V3: VALIDATION REPORT ===")
print(classification_report(y_val, val_pred_rf_v3, digits=4))
print("ROC-AUC:", roc_auc_score(y_val, val_proba_rf_v3))
print("PR-AUC :", average_precision_score(y_val, val_proba_rf_v3))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred_rf_v3))

Fitting Random Forest v3 (numeric only)...

=== RANDOM FOREST V3: VALIDATION REPORT ===
              precision    recall  f1-score   support

           0     0.9888    0.9628    0.9756   2336002
           1     0.9176    0.9742    0.9451    992519

    accuracy                         0.9662   3328521
   macro avg     0.9532    0.9685    0.9603   3328521
weighted avg     0.9675    0.9662    0.9665   3328521

ROC-AUC: 0.9966262896993725
PR-AUC : 0.9923961621429943
Confusion Matrix:
 [[2249150   86852]
 [  25583  966936]]


In [56]:
results["rf_v3_numeric_only"] = {
    "roc_auc": roc_auc_score(y_val, val_proba_rf_v3),
    "pr_auc": average_precision_score(y_val, val_proba_rf_v3),
    "report": classification_report(y_val, val_pred_rf_v3, digits=4),
    "confusion_matrix": confusion_matrix(y_val, val_pred_rf_v3),
}

print("=== COMPARISON ===")
print("V1 ROC-AUC:", results["rf_v1_with_vehicle_id"]["roc_auc"])
print("V2 ROC-AUC:", results["rf_v2_without_vehicle_id"]["roc_auc"])
print("V3 ROC-AUC:", results["rf_v3_numeric_only"]["roc_auc"])

print("V1 PR-AUC :", results["rf_v1_with_vehicle_id"]["pr_auc"])
print("V2 PR-AUC :", results["rf_v2_without_vehicle_id"]["pr_auc"])
print("V3 PR-AUC :", results["rf_v3_numeric_only"]["pr_auc"])

=== COMPARISON ===
V1 ROC-AUC: 0.9939707244167337
V2 ROC-AUC: 0.9956153748828321
V3 ROC-AUC: 0.9966262896993725
V1 PR-AUC : 0.9860831255255261
V2 PR-AUC : 0.989837546818113
V3 PR-AUC : 0.9923961621429943


# Trying with future_late_3min_300s rather than future_late_3min_120s
i.e. with 5 minutes window rather than 2 minutes

In [20]:
import pandas as pd
import numpy as np

# ----------------------------
# SETTINGS
# ----------------------------
HORIZON_SEC = 300
GROUP_KEYS = ["vehicle_id", "stop_id", "direction", "destination_name", "run_id"]

# We use the existing df that already has run_id
df = df.sort_values(GROUP_KEYS + ["observed_at"]).reset_index(drop=True)

# If row_id already exists, keep it; otherwise create it
if "row_id" not in df.columns:
    df["row_id"] = np.arange(len(df))

future_parts_300 = []

for key, g in df.groupby(GROUP_KEYS, sort=False):
    g = g.sort_values("observed_at").copy()

    g["target_time_300s"] = g["observed_at"] + pd.Timedelta(seconds=HORIZON_SEC)

    right = g[["observed_at", "late_3min"]].copy()
    right = right.rename(columns={
        "observed_at": "future_observed_at_300s",
        "late_3min": "future_late_3min_300s"
    })

    matched = pd.merge_asof(
        g.sort_values("target_time_300s"),
        right.sort_values("future_observed_at_300s"),
        left_on="target_time_300s",
        right_on="future_observed_at_300s",
        direction="forward"
    )

    future_parts_300.append(
        matched[["row_id", "target_time_300s", "future_observed_at_300s", "future_late_3min_300s"]]
    )

future_df_300 = pd.concat(future_parts_300, ignore_index=True)

# Merge back
df_300 = df.merge(future_df_300, on="row_id", how="left")

# Keep only rows with valid future target
df_path_b_300 = df_300.dropna(subset=["future_late_3min_300s"]).copy()
df_path_b_300["future_late_3min_300s"] = df_path_b_300["future_late_3min_300s"].astype(int)

print("Original rows:", len(df))
print("300s rows with valid future target:", len(df_path_b_300))
print("Missing fraction:", df_300["future_late_3min_300s"].isna().mean())
print(df_path_b_300["future_late_3min_300s"].value_counts(dropna=False))

Original rows: 24571322
300s rows with valid future target: 18867608
Missing fraction: 0.2321289021404709
future_late_3min_300s
0    14502167
1     4365441
Name: count, dtype: int64


In [21]:
df_path_b_300 = df_path_b_300.sort_values("observed_at")

t1_300 = df_path_b_300["observed_at"].quantile(0.7)
t2_300 = df_path_b_300["observed_at"].quantile(0.85)

train_df_300 = df_path_b_300[df_path_b_300["observed_at"] < t1_300]
val_df_300   = df_path_b_300[(df_path_b_300["observed_at"] >= t1_300) & (df_path_b_300["observed_at"] < t2_300)]
test_df_300  = df_path_b_300[df_path_b_300["observed_at"] >= t2_300]

print("t1_300 =", t1_300)
print("t2_300 =", t2_300)
print("train_300:", len(train_df_300))
print("val_300:", len(val_df_300))
print("test_300:", len(test_df_300))

t1_300 = 2026-03-02 17:19:49.688857088+00:00
t2_300 = 2026-03-03 22:58:48.026457088+00:00
train_300: 13207497
val_300: 2830085
test_300: 2830026


In [59]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

target_col_300 = "future_late_3min_300s"

numeric_features_v3 = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_300 = train_df_300[numeric_features_v3].copy()
y_train_300 = train_df_300[target_col_300].copy()

X_val_300 = val_df_300[numeric_features_v3].copy()
y_val_300 = val_df_300[target_col_300].copy()

numeric_pipeline_v3_300 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

print("Fitting Random Forest v3 (numeric only, 300s horizon)...")
numeric_pipeline_v3_300.fit(X_train_300, y_train_300)

val_proba_rf_v3_300 = numeric_pipeline_v3_300.predict_proba(X_val_300)[:, 1]
val_pred_rf_v3_300 = (val_proba_rf_v3_300 >= 0.5).astype(int)

print("\n=== RANDOM FOREST V3: VALIDATION REPORT (300s) ===")
print(classification_report(y_val_300, val_pred_rf_v3_300, digits=4))
print("ROC-AUC:", roc_auc_score(y_val_300, val_proba_rf_v3_300))
print("PR-AUC :", average_precision_score(y_val_300, val_proba_rf_v3_300))
print("Confusion Matrix:\n", confusion_matrix(y_val_300, val_pred_rf_v3_300))

Fitting Random Forest v3 (numeric only, 300s horizon)...

=== RANDOM FOREST V3: VALIDATION REPORT (300s) ===
              precision    recall  f1-score   support

           0     0.9813    0.9530    0.9669   2213838
           1     0.8469    0.9348    0.8887    616247

    accuracy                         0.9490   2830085
   macro avg     0.9141    0.9439    0.9278   2830085
weighted avg     0.9520    0.9490    0.9499   2830085

ROC-AUC: 0.9881889470218322
PR-AUC : 0.9675782738895131
Confusion Matrix:
 [[2109708  104130]
 [  40174  576073]]


## Features for 300s V2 (numeric + categorical without vehicle_id)

In [60]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

target_col_300 = "future_late_3min_300s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

categorical_features = [
    "stop_id",
    "line_id",
    "direction",
    "destination_name",
    "platform_name",
]

features_v2_300 = numeric_features + categorical_features

X_train_300 = train_df_300[features_v2_300].copy()
y_train_300 = train_df_300[target_col_300].copy()

X_val_300 = val_df_300[features_v2_300].copy()
y_val_300 = val_df_300[target_col_300].copy()


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced_subsample"
)

pipeline_v2_300 = Pipeline([
    ("prep", preprocessor),
    ("model", rf_model),
])

print("Fitting Random Forest V2 (numeric + categorical, 300s)...")
pipeline_v2_300.fit(X_train_300, y_train_300)

val_proba_v2_300 = pipeline_v2_300.predict_proba(X_val_300)[:, 1]
val_pred_v2_300 = (val_proba_v2_300 >= 0.5).astype(int)

print("\n=== RANDOM FOREST V2 (300s) ===")
print(classification_report(y_val_300, val_pred_v2_300, digits=4))
print("ROC-AUC:", roc_auc_score(y_val_300, val_proba_v2_300))
print("PR-AUC :", average_precision_score(y_val_300, val_proba_v2_300))
print("Confusion Matrix:\n", confusion_matrix(y_val_300, val_pred_v2_300))

Fitting Random Forest V2 (numeric + categorical, 300s)...

=== RANDOM FOREST V2 (300s) ===
              precision    recall  f1-score   support

           0     0.9818    0.9493    0.9653   2213838
           1     0.8372    0.9369    0.8842    616247

    accuracy                         0.9466   2830085
   macro avg     0.9095    0.9431    0.9248   2830085
weighted avg     0.9503    0.9466    0.9476   2830085

ROC-AUC: 0.9857402923037671
PR-AUC : 0.9639946427470647
Confusion Matrix:
 [[2101524  112314]
 [  38861  577386]]


In [61]:
print("=== 300s COMPARISON ===")

print("Numeric only ROC:", roc_auc_score(y_val_300, val_proba_rf_v3_300))
print("Numeric+Cat ROC:", roc_auc_score(y_val_300, val_proba_v2_300))

print("Numeric only PR :", average_precision_score(y_val_300, val_proba_rf_v3_300))
print("Numeric+Cat PR :", average_precision_score(y_val_300, val_proba_v2_300))

=== 300s COMPARISON ===
Numeric only ROC: 0.9881889470218322
Numeric+Cat ROC: 0.9857402923037671
Numeric only PR : 0.9675782738895131
Numeric+Cat PR : 0.9639946427470647


## Another check using Horizon = 1800 seconds or 30 minutes

In [23]:
import pandas as pd
import numpy as np

HORIZON_SEC = 1800

GROUP_KEYS = [
    "vehicle_id",
    "stop_id",
    "direction",
    "destination_name",
    "run_id",
]

df = df.sort_values(GROUP_KEYS + ["observed_at"]).reset_index(drop=True)

if "row_id" not in df.columns:
    df["row_id"] = np.arange(len(df))


future_parts_1800 = []

for key, g in df.groupby(GROUP_KEYS, sort=False):

    g = g.sort_values("observed_at").copy()

    g["target_time_1800s"] = (
        g["observed_at"] + pd.Timedelta(seconds=HORIZON_SEC)
    )

    right = g[["observed_at", "late_3min"]].copy()

    right = right.rename(
        columns={
            "observed_at": "future_observed_at_1800s",
            "late_3min": "future_late_3min_1800s",
        }
    )

    matched = pd.merge_asof(
        g.sort_values("target_time_1800s"),
        right.sort_values("future_observed_at_1800s"),
        left_on="target_time_1800s",
        right_on="future_observed_at_1800s",
        direction="forward",
    )

    future_parts_1800.append(
        matched[
            [
                "row_id",
                "target_time_1800s",
                "future_observed_at_1800s",
                "future_late_3min_1800s",
            ]
        ]
    )


future_df_1800 = pd.concat(future_parts_1800, ignore_index=True)

df_1800 = df.merge(future_df_1800, on="row_id", how="left")

df_path_b_1800 = df_1800.dropna(
    subset=["future_late_3min_1800s"]
).copy()

df_path_b_1800["future_late_3min_1800s"] = (
    df_path_b_1800["future_late_3min_1800s"].astype(int)
)

print("Original rows:", len(df))
print("Valid rows 1800s:", len(df_path_b_1800))
print(
    "Missing fraction:",
    df_1800["future_late_3min_1800s"].isna().mean(),
)

print(
    df_path_b_1800["future_late_3min_1800s"].value_counts()
)

Original rows: 24571322
Valid rows 1800s: 2182397
Missing fraction: 0.9111811322158408
future_late_3min_1800s
0    1430946
1     751451
Name: count, dtype: int64


In [24]:
df_path_b_1800 = df_path_b_1800.sort_values("observed_at")

t1_1800 = df_path_b_1800["observed_at"].quantile(0.7)
t2_1800 = df_path_b_1800["observed_at"].quantile(0.85)

train_df_1800 = df_path_b_1800[
    df_path_b_1800["observed_at"] < t1_1800
]

val_df_1800 = df_path_b_1800[
    (df_path_b_1800["observed_at"] >= t1_1800)
    & (df_path_b_1800["observed_at"] < t2_1800)
]

test_df_1800 = df_path_b_1800[
    df_path_b_1800["observed_at"] >= t2_1800
]

print("t1:", t1_1800)
print("t2:", t2_1800)

print("train:", len(train_df_1800))
print("val:", len(val_df_1800))
print("test:", len(test_df_1800))

t1: 2026-03-04 09:22:46.266557952+00:00
t2: 2026-03-04 12:31:05.987824128+00:00
train: 1527604
val: 327525
test: 327268


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col_1800 = "future_late_3min_1800s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_1800 = train_df_1800[numeric_features]
y_train_1800 = train_df_1800[target_col_1800]

X_val_1800 = val_df_1800[numeric_features]
y_val_1800 = val_df_1800[target_col_1800]


pipeline_1800 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=20,
                n_jobs=-1,
                random_state=42,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

print("Fitting RF 1800s...")
pipeline_1800.fit(X_train_1800, y_train_1800)

proba_1800 = pipeline_1800.predict_proba(X_val_1800)[:, 1]
pred_1800 = (proba_1800 >= 0.5).astype(int)

print("\n=== 1800s RESULT ===")

print(classification_report(y_val_1800, pred_1800, digits=4))

print("ROC:", roc_auc_score(y_val_1800, proba_1800))
print("PR :", average_precision_score(y_val_1800, proba_1800))

print(
    confusion_matrix(y_val_1800, pred_1800)
)

Fitting RF 1800s...

=== 1800s RESULT ===
              precision    recall  f1-score   support

           0     0.6750    0.9942    0.8041    199985
           1     0.9649    0.2494    0.3963    127540

    accuracy                         0.7042    327525
   macro avg     0.8200    0.6218    0.6002    327525
weighted avg     0.7879    0.7042    0.6453    327525

ROC: 0.9815158121922655
PR : 0.952949123930434
[[198829   1156]
 [ 95737  31803]]


## Logistic Regression for horizon = 5 minutes

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col = "future_late_3min_300s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train = train_df_300[numeric_features]
y_train = train_df_300[target_col]

X_val = val_df_300[numeric_features]
y_val = val_df_300[target_col]


log_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=200,
                n_jobs=-1,
            ),
        ),
    ]
)

print("Fitting Logistic Regression...")

log_pipeline.fit(X_train, y_train)

proba_log = log_pipeline.predict_proba(X_val)[:, 1]
pred_log = (proba_log >= 0.5).astype(int)

print("\n=== LOGISTIC RESULT (300s) ===")

print(classification_report(y_val, pred_log, digits=4))

print("ROC:", roc_auc_score(y_val, proba_log))
print("PR :", average_precision_score(y_val, proba_log))

print(confusion_matrix(y_val, pred_log))

Fitting Logistic Regression...


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



=== LOGISTIC RESULT (300s) ===
              precision    recall  f1-score   support

           0     0.9659    0.9780    0.9719   2213838
           1     0.9173    0.8759    0.8961    616247

    accuracy                         0.9558   2830085
   macro avg     0.9416    0.9269    0.9340   2830085
weighted avg     0.9553    0.9558    0.9554   2830085

ROC: 0.986883292161785
PR : 0.9642243828241688
[[2165166   48672]
 [  76505  539742]]


### Logistic Regression for horizon = 30 minutes

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col_1800 = "future_late_3min_1800s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_1800 = train_df_1800[numeric_features].copy()
y_train_1800 = train_df_1800[target_col_1800].copy()

X_val_1800 = val_df_1800[numeric_features].copy()
y_val_1800 = val_df_1800[target_col_1800].copy()

log_pipeline_1800 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=300,
            class_weight="balanced",
            random_state=42
        )),
    ]
)

print("Fitting Logistic Regression 1800s...")
log_pipeline_1800.fit(X_train_1800, y_train_1800)

proba_log_1800 = log_pipeline_1800.predict_proba(X_val_1800)[:, 1]
pred_log_1800 = (proba_log_1800 >= 0.5).astype(int)

print("\n=== LOGISTIC RESULT (1800s) ===")
print(classification_report(y_val_1800, pred_log_1800, digits=4))
print("ROC:", roc_auc_score(y_val_1800, proba_log_1800))
print("PR :", average_precision_score(y_val_1800, proba_log_1800))
print(confusion_matrix(y_val_1800, pred_log_1800))

Fitting Logistic Regression 1800s...

=== LOGISTIC RESULT (1800s) ===
              precision    recall  f1-score   support

           0     0.8898    0.9991    0.9413    199985
           1     0.9982    0.8059    0.8918    127540

    accuracy                         0.9239    327525
   macro avg     0.9440    0.9025    0.9165    327525
weighted avg     0.9320    0.9239    0.9220    327525

ROC: 0.9974874995623104
PR : 0.9944278884894173
[[199796    189]
 [ 24750 102790]]


## LightGBM for horizon = 5 minutes

In [26]:
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [27]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

target_col_300 = "future_late_3min_300s"

numeric_features = [
    "hour",
    "weekday",
    "is_weekend",
    "time_to_station",
    "roll_mean_tts_10m",
    "roll_max_tts_10m",
    "roll_count_10m",
    "baseline_median_tts",
    "deviation_from_baseline",
]

X_train_300 = train_df_300[numeric_features].copy()
y_train_300 = train_df_300[target_col_300].copy()

X_val_300 = val_df_300[numeric_features].copy()
y_val_300 = val_df_300[target_col_300].copy()

lgbm_pipeline_300 = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
            verbose=-1
        )),
    ]
)

print("Fitting LightGBM 300s (CPU)...")
lgbm_pipeline_300.fit(X_train_300, y_train_300)

proba_lgbm_300 = lgbm_pipeline_300.predict_proba(X_val_300)[:, 1]
pred_lgbm_300 = (proba_lgbm_300 >= 0.5).astype(int)

print("\n=== LIGHTGBM RESULT (300s, CPU) ===")
print(classification_report(y_val_300, pred_lgbm_300, digits=4))
print("ROC:", roc_auc_score(y_val_300, proba_lgbm_300))
print("PR :", average_precision_score(y_val_300, proba_lgbm_300))
print(confusion_matrix(y_val_300, pred_lgbm_300))

Fitting LightGBM 300s (CPU)...


e:\dsbootcamp_neuefische\capstone_project\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== LIGHTGBM RESULT (300s, CPU) ===
              precision    recall  f1-score   support

           0     0.9814    0.9516    0.9663   2213838
           1     0.8433    0.9351    0.8868    616247

    accuracy                         0.9480   2830085
   macro avg     0.9123    0.9434    0.9265   2830085
weighted avg     0.9513    0.9480    0.9490   2830085

ROC: 0.988025582370121
PR : 0.9666764863541033
[[2106779  107059]
 [  40019  576228]]


#### LightGBM with GPU

In [28]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

lgbm_pipeline_300_gpu = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
            verbose=-1,
            device="gpu"
        )),
    ]
)

print("Fitting LightGBM 300s (GPU)...")
lgbm_pipeline_300_gpu.fit(X_train_300, y_train_300)

Fitting LightGBM 300s (GPU)...


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

In [30]:
print("=== MODEL COMPARISON ===")


print("LOG 300 ROC:", roc_auc_score(y_val_300, proba_log))
print("LOG 300 PR :", average_precision_score(y_val_300, proba_log))

print("LGBM 300 ROC:", roc_auc_score(y_val_300, proba_lgbm_300))
print("LGBM 300 PR :", average_precision_score(y_val_300, proba_lgbm_300))



print("LOG 1800 ROC:", roc_auc_score(y_val_1800, proba_log_1800))
print("LOG 1800 PR :", average_precision_score(y_val_1800, proba_log_1800))

=== MODEL COMPARISON ===
LOG 300 ROC: 0.986883292161785
LOG 300 PR : 0.9642243828241688
LGBM 300 ROC: 0.988025582370121
LGBM 300 PR : 0.9666764863541033
LOG 1800 ROC: 0.9974874995623104
LOG 1800 PR : 0.9944278884894173


In [31]:
print("X_train_1800:", X_train_1800.shape)
print("y_train_1800:", y_train_1800.shape)
print("X_val_1800:", X_val_1800.shape)
print("y_val_1800:", y_val_1800.shape)

print("Target train distribution:")
print(y_train_1800.value_counts(dropna=False))

print("Target val distribution:")
print(y_val_1800.value_counts(dropna=False))

X_train_1800: (1527604, 9)
y_train_1800: (1527604,)
X_val_1800: (327525, 9)
y_val_1800: (327525,)
Target train distribution:
future_late_3min_1800s
0    1027514
1     500090
Name: count, dtype: int64
Target val distribution:
future_late_3min_1800s
0    199985
1    127540
Name: count, dtype: int64


In [32]:
print("LOG 1800 ROC:", roc_auc_score(y_val_1800, proba_log_1800))
print("LOG 1800 PR :", average_precision_score(y_val_1800, proba_log_1800))

LOG 1800 ROC: 0.9974874995623104
LOG 1800 PR : 0.9944278884894173


In [33]:
pred_log_1800 = (proba_log_1800 >= 0.5).astype(int)

print(classification_report(y_val_1800, pred_log_1800, digits=4))
print(confusion_matrix(y_val_1800, pred_log_1800))

              precision    recall  f1-score   support

           0     0.8898    0.9991    0.9413    199985
           1     0.9982    0.8059    0.8918    127540

    accuracy                         0.9239    327525
   macro avg     0.9440    0.9025    0.9165    327525
weighted avg     0.9320    0.9239    0.9220    327525

[[199796    189]
 [ 24750 102790]]
